In [18]:
import pymysql
import requests
import json

###　下載分析pm2.5 opendata資料

In [19]:
url="https://data.moenv.gov.tw/api/v2/aqx_p_02?api_key=af57253c-e838-46da-a1f5-12b43afd75f3&limit=1000&sort=datacreationdate%20desc&format=JSON"

In [20]:
headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
res=requests.get(url,headers=headers,verify=False)
res



c:\Users\roger\Desktop\Django-陳岳洋\5.MYSQL\pm25_tomysql複習2\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'data.moenv.gov.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


<Response [200]>

In [22]:
datas=res.json()

In [23]:
len(datas)

1000

In [24]:
datas[0]

{'site': '林森',
 'county': '臺南市',
 'pm25': '24',
 'datacreationdate': '2026-04-21 03:00',
 'itemunit': 'μg/m3'}

### 建立資料表

In [10]:
table_str="""
create table if not exists pm25(
    id int auto_increment primary key,
    site varchar(25),
    county varchar(25),
    pm25 int,
    datacreationdate datetime,
    itemunit varchar(20),
    unique key site_time (site, datacreationdate)
)
"""

### 連接資料庫

In [25]:
conn = pymysql.connect(
    host="mysql-3650ed2-pm25.e.aivencloud.com",
    user="avnadmin",
    passwd="AVNS_-hDOnmEfwqkcZfFy7WF",
    port=21697,
    db="defaultdb",
)

# conn = pymysql.connect(
#     host="127.0.0.1",
#     user="root",
#     passwd="",
#     port=3307,
#     db="demo",
# )
cursor=conn.cursor()
cursor

In [11]:
cursor.execute(table_str)
conn.commit()

### 寫入一筆資料

In [12]:
sqlstr="insert into pm25(site,county,pm25,datacreationdate,itemunit)\
 values('{}', '{}', '{}', '{}', '{}')"
data=list(datas[0].values())
print(data)
sqlstr.format(data[0]  ,data[1]  ,data[2]  ,data[3]  ,data[4]  )

['林森', '臺南市', '20', '2026-04-12 00:00', 'μg/m3']


"insert into pm25(site,county,pm25,datacreationdate,itemunit) values('林森', '臺南市', '20', '2026-04-12 00:00', 'μg/m3')"

In [13]:
cursor.execute(sqlstr.format(data[0]  ,data[1]  ,data[2]  ,data[3]  ,data[4]  ))
conn.commit()

### 一次寫入
- 加上 ignore

In [15]:
sqlstr="insert ignore into pm25(site,county,pm25,datacreationdate,itemunit)\
 values(%s, %s, %s, %s, %s)"

# 移除pm25空字串部分
values=[list(data.values()) for data in datas if list(data.values())[2]!=""]

cursor.executemany(sqlstr,values)
conn.commit()

In [16]:
conn.close()

In [16]:
import pandas as pd

In [36]:
cursor.execute("select * from pm25")

datas=cursor.fetchall()
datas

((1, '林森', '臺南市', 20, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (2, '員林', '彰化縣', 21, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (3, '臺灣大道', '臺中市', 23, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (4, '大城', '彰化縣', 23, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (5, '富貴角', '新北市', 16, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (6, '麥寮', '雲林縣', 23, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (7, '關山', '臺東縣', 17, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (8, '金門', '金門縣', 13, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (9, '馬祖', '連江縣', 17, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (10, '埔里', '南投縣', 23, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (11, '復興', '高雄市', 17, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (12, '永和', '新北市', 21, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (13, '竹山', '南投縣', 19, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (14, '中壢', '桃園市', 21, datetime.datetime(2026, 4, 12, 0, 0), 'μg/m3'),
 (15, '三重', 

In [37]:
df=pd.DataFrame(datas,columns=["id","site","county","pm25","datetime","itemunit"])
df

,id,site,county,pm25,datetime,itemunit
0,1,林森,臺南市,20,2026-04-12 00:00:00,μg/m3
1,2,員林,彰化縣,21,2026-04-12 00:00:00,μg/m3
2,3,臺灣大道,臺中市,23,2026-04-12 00:00:00,μg/m3
3,4,大城,彰化縣,23,2026-04-12 00:00:00,μg/m3
4,5,富貴角,新北市,16,2026-04-12 00:00:00,μg/m3
...,...,...,...,...,...,...
989,990,臺南,臺南市,17,2026-04-11 12:00:00,μg/m3
990,991,安南,臺南市,19,2026-04-11 12:00:00,μg/m3
991,992,善化,臺南市,20,2026-04-11 12:00:00,μg/m3
992,993,新營,臺南市,20,2026-04-11 12:00:00,μg/m3


In [38]:
df.groupby("county").get_group("新北市")

,id,site,county,pm25,datetime,itemunit
4,5,富貴角,新北市,16,2026-04-12 00:00:00,μg/m3
11,12,永和,新北市,21,2026-04-12 00:00:00,μg/m3
14,15,三重,新北市,19,2026-04-12 00:00:00,μg/m3
70,71,淡水,新北市,23,2026-04-12 00:00:00,μg/m3
71,72,林口,新北市,23,2026-04-12 00:00:00,μg/m3
...,...,...,...,...,...,...
951,952,新店,新北市,25,2026-04-11 13:00:00,μg/m3
952,953,汐止,新北市,26,2026-04-11 13:00:00,μg/m3
958,959,富貴角,新北市,15,2026-04-11 12:00:00,μg/m3
966,967,永和,新北市,23,2026-04-11 12:00:00,μg/m3


In [40]:
df[["site","datetime"]].drop_duplicates()

,site,datetime
0,林森,2026-04-12 00:00:00
1,員林,2026-04-12 00:00:00
2,臺灣大道,2026-04-12 00:00:00
3,大城,2026-04-12 00:00:00
4,富貴角,2026-04-12 00:00:00
...,...,...
989,臺南,2026-04-11 12:00:00
990,安南,2026-04-11 12:00:00
991,善化,2026-04-11 12:00:00
992,新營,2026-04-11 12:00:00


### 取得各縣市站點的平均值

In [3]:
cursor.execute("select avg(pm25) from pm25;")
float(cursor.fetchone()[0])

19.0987

In [ ]:
sqlstr="""
select county,round(avg(pm25),2) from pm25 group by county;
"""

cursor.execute(sqlstr)
cursor.fetchall()

(('臺南市', Decimal('16.18')),
 ('彰化縣', Decimal('17.81')),
 ('臺中市', Decimal('16.91')),
 ('新北市', Decimal('21.03')),
 ('雲林縣', Decimal('17.39')),
 ('臺東縣', Decimal('18.29')),
 ('金門縣', Decimal('15.57')),
 ('連江縣', Decimal('25.64')),
 ('南投縣', Decimal('18.91')),
 ('高雄市', Decimal('17.84')),
 ('桃園市', Decimal('17.63')),
 ('宜蘭縣', Decimal('20.25')),
 ('臺北市', Decimal('20.13')),
 ('花蓮縣', Decimal('20.68')),
 ('屏東縣', Decimal('17.41')),
 ('嘉義市', Decimal('16.61')),
 ('嘉義縣', Decimal('16.30')),
 ('苗栗縣', Decimal('16.60')),
 ('新竹市', Decimal('14.19')),
 ('新竹縣', Decimal('17.13')),
 ('基隆市', Decimal('19.12')))

In [ ]:
df.groupby("county")["pm25"].mean()

In [10]:
cursor.execute("select avg(pm25) from pm25")
cursor.fetchone()[0]

Decimal('19.0987')

In [25]:
sqlstr="""
select county,round(avg(pm25),2) from pm25 group by county
"""
cursor.execute(sqlstr)
cursor.fetchall()

(('臺南市', Decimal('19.48')),
 ('彰化縣', Decimal('21.30')),
 ('臺中市', Decimal('19.27')),
 ('新北市', Decimal('18.42')),
 ('雲林縣', Decimal('22.90')),
 ('臺東縣', Decimal('17.99')),
 ('金門縣', Decimal('18.29')),
 ('連江縣', Decimal('23.78')),
 ('南投縣', Decimal('21.40')),
 ('高雄市', Decimal('19.35')),
 ('桃園市', Decimal('17.17')),
 ('宜蘭縣', Decimal('17.57')),
 ('臺北市', Decimal('17.79')),
 ('花蓮縣', Decimal('17.51')),
 ('屏東縣', Decimal('18.14')),
 ('嘉義市', Decimal('21.05')),
 ('嘉義縣', Decimal('22.19')),
 ('苗栗縣', Decimal('17.21')),
 ('新竹市', Decimal('15.53')),
 ('新竹縣', Decimal('18.07')),
 ('基隆市', Decimal('17.34')),
 ('澎湖縣', Decimal('16.69')))

In [27]:
conn.close()

### 取得單一縣市的站點數值

In [34]:
county="新北市"

#取得最新資料
sqlstr="""
select site,pm25,datacreationdate from pm25
where county=%s
and datacreationdate=(select max(datacreationdate) from pm25);
"""

cursor.execute(sqlstr,(county,))
cursor.fetchall()

(('富貴角', 18, datetime.datetime(2026, 4, 21, 3, 0)),
 ('永和', 10, datetime.datetime(2026, 4, 21, 3, 0)),
 ('三重', 14, datetime.datetime(2026, 4, 21, 3, 0)),
 ('淡水', 13, datetime.datetime(2026, 4, 21, 3, 0)),
 ('林口', 12, datetime.datetime(2026, 4, 21, 3, 0)),
 ('菜寮', 13, datetime.datetime(2026, 4, 21, 3, 0)),
 ('新莊', 13, datetime.datetime(2026, 4, 21, 3, 0)),
 ('板橋', 12, datetime.datetime(2026, 4, 21, 3, 0)),
 ('土城', 10, datetime.datetime(2026, 4, 21, 3, 0)),
 ('新店', 14, datetime.datetime(2026, 4, 21, 3, 0)),
 ('汐止', 16, datetime.datetime(2026, 4, 21, 3, 0)))